In [1]:
# ============================================
# SILVER LAYER: DATA CLEANING + METRICS
# ============================================

from pyspark.sql.functions import *
import time, json


StatementMeta(, 4f5a8a40-3b2d-4753-98bf-f3a867fc5ed0, 3, Finished, Available, Finished, False)

In [2]:

# ================================
# ADDED: PERFORMANCE SETUP
# ================================
metrics = {}
start_time = time.time()

def log_metric(name, value):
    metrics[name] = value

def count_df(df):
    return df.count()


StatementMeta(, 4f5a8a40-3b2d-4753-98bf-f3a867fc5ed0, 4, Finished, Available, Finished, False)

In [3]:
Orders_df = spark.read.parquet("Files/ShoppingMart_Silver_Orders/ShoppingMart_customers_orderdata")

StatementMeta(, 4f5a8a40-3b2d-4753-98bf-f3a867fc5ed0, 5, Finished, Available, Finished, False)

In [4]:
reviews_df = spark.read.parquet("Files/ShoppingMart_Silver_Reviews/ShoppingMart_review")
social_df = spark.read.parquet("Files/ShoppingMart_Silver_Social_Media/ShoppingMart_social_media")
weblogs_df = spark.read.parquet("Files/ShoppingMart_Silver_Web_Logs/ShoppingMart_web_logs")
display(social_df)

StatementMeta(, 4f5a8a40-3b2d-4753-98bf-f3a867fc5ed0, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d0cdfaf5-4fc6-40f5-9e85-d306e2395d6a)

In [5]:
# ================================
# ADDED: INPUT METRICS
# ================================
log_metric("orders_input_rows", count_df(Orders_df))
log_metric("reviews_input_rows", count_df(reviews_df))
log_metric("social_input_rows", count_df(social_df))
log_metric("weblogs_input_rows", count_df(weblogs_df))

weblogs_before = count_df(weblogs_df)

StatementMeta(, 4f5a8a40-3b2d-4753-98bf-f3a867fc5ed0, 7, Finished, Available, Finished, False)

In [6]:
# KPI1 : Aggregates web log data to measure engagement per user on each page and action.
weblogs_df = spark.read.parquet("Files/ShoppingMart_Silver_Web_Logs/ShoppingMart_web_logs")
weblogs_df = weblogs_df.groupBy("user_id", "page", "action").count()

StatementMeta(, 4f5a8a40-3b2d-4753-98bf-f3a867fc5ed0, 8, Finished, Available, Finished, False)

In [7]:

weblogs_df.write.mode("overwrite").parquet("Files/ShoppingMart_Gold_Web_Logs/ShoppingMart_web_logs")
display(weblogs_df)

StatementMeta(, 4f5a8a40-3b2d-4753-98bf-f3a867fc5ed0, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4395e9f0-1009-44f2-83fa-e543d6b98fc1)

In [8]:
# ================================
# ADDED: KPI1 METRICS
# ================================
weblogs_df.cache()
weblogs_after = count_df(weblogs_df)
log_metric("weblogs_reduction_ratio", weblogs_before / weblogs_after if weblogs_after else 0)

social_before = count_df(social_df)

StatementMeta(, 4f5a8a40-3b2d-4753-98bf-f3a867fc5ed0, 10, Finished, Available, Finished, False)

In [9]:
# KPI2 : Aggregates unstructured social media data to track sentiment trends across different platforms.
social_df = spark.read.parquet("Files/ShoppingMart_Silver_Social_Media/ShoppingMart_social_media")
social_df= social_df.groupBy("platform","sentiment" ).count()
social_df.write.mode("overwrite").parquet("Files/ShoppingMart_Gold_Social_Media/ShoppingMart_social_media")
display(social_df)

StatementMeta(, 4f5a8a40-3b2d-4753-98bf-f3a867fc5ed0, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e07588b5-e4ba-4dd6-8b14-3090d8ef755c)

In [11]:
# ================================
# ADDED: KPI2 METRICS
# ================================
social_df.cache()
social_after = count_df(social_df)
log_metric("social_reduction_ratio", social_before / social_after if social_after else 0)

reviews_before = count_df(reviews_df)

StatementMeta(, 4f5a8a40-3b2d-4753-98bf-f3a867fc5ed0, 13, Finished, Available, Finished, False)

In [12]:
#KPI3: Aggregates product reviews to calculate the average rating per product.
reviews_df = spark.read.parquet("Files/ShoppingMart_Silver_Reviews/ShoppingMart_review")
reviews_df = reviews_df.groupBy("product_id").agg(avg("rating").alias("AvgRating"))
reviews_df.write.mode("overwrite").parquet("Files/ShoppingMart_Gold_Reviews/ShoppingMart_review")
display(reviews_df)

StatementMeta(, 4f5a8a40-3b2d-4753-98bf-f3a867fc5ed0, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7240b520-4234-464d-a397-d2030c123a07)

In [13]:

# ================================
# ADDED: KPI3 METRICS
# ================================
reviews_df.cache()
reviews_after = count_df(reviews_df)
log_metric("reviews_reduction_ratio", reviews_before / reviews_after if reviews_after else 0)


StatementMeta(, 4f5a8a40-3b2d-4753-98bf-f3a867fc5ed0, 15, Finished, Available, Finished, False)

In [14]:
Orders_df.write.mode("overwrite").parquet("Files/ShoppingMart_Gold_Orders/ShoppingMart_customers_orderdata")

StatementMeta(, 4f5a8a40-3b2d-4753-98bf-f3a867fc5ed0, 16, Finished, Available, Finished, False)

In [15]:

# ================================
# ADDED: FINAL PERFORMANCE
# ================================
end_time = time.time()

execution_time = end_time - start_time

total_output_rows = (
    count_df(weblogs_df) +
    count_df(social_df) +
    count_df(reviews_df)
)

throughput = total_output_rows / execution_time if execution_time > 0 else 0

log_metric("gold_execution_time_sec", execution_time)
log_metric("gold_output_rows", total_output_rows)
log_metric("gold_throughput_rows_per_sec", throughput)

spark.createDataFrame([(json.dumps(metrics),)], ["metrics"]) \
    .write.mode("append") \
    .text("Files/ETL_Metrics/Gold_metrics")

print("✅ Gold metrics captured")

StatementMeta(, 4f5a8a40-3b2d-4753-98bf-f3a867fc5ed0, 17, Finished, Available, Finished, False)

✅ Gold metrics captured


In [16]:
# ================================
# PRINT GOLD TABLE SCHEMAS
# ================================

print("===== GOLD TABLE SCHEMAS =====\n")

gold_paths = {
    "Orders_df (Gold Orders)": "Files/ShoppingMart_Gold_Orders/ShoppingMart_customers_orderdata",
    "reviews_df (Gold Reviews)": "Files/ShoppingMart_Gold_Reviews/ShoppingMart_review",
    "social_df (Gold Social)": "Files/ShoppingMart_Gold_Social_Media/ShoppingMart_social_media",
    "weblogs_df (Gold Weblogs)": "Files/ShoppingMart_Gold_Web_Logs/ShoppingMart_web_logs"
}

for name, path in gold_paths.items():
    print(f"\n--- {name} ---")
    df = spark.read.parquet(path)
    df.printSchema()

StatementMeta(, 4f5a8a40-3b2d-4753-98bf-f3a867fc5ed0, 18, Finished, Available, Finished, False)

===== GOLD TABLE SCHEMAS =====


--- Orders_df (Gold Orders) ---
root
 |-- ProductID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- OrderID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- TotalAmount: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- SignupDate: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Stock: string (nullable = true)
 |-- UnitPrice: string (nullable = true)


--- reviews_df (Gold Reviews) ---
root
 |-- product_id: string (nullable = true)
 |-- AvgRating: double (nullable = true)


--- social_df (Gold Social) ---
root
 |-- platform: string (nullable = true)
 |-- sentiment: string (nullable = true)
 |-- count: long (nullable = true)


--- weblogs_df (Gold Weblogs) ---
ro